# Parse ICD10 codes from CMS

In [3]:
# download ICD codes from CMS as an .xml file
# source: https://www.cms.gov/medicare/coding-billing/icd-10-codes
import pandas as pd
import xml.dom.minidom
df = [] # results container
doc = xml.dom.minidom.parse('icd10cm_tabular_2026.xml')
# the .xml file has nested structure (chapter, section, diag, subdiag)
chapters = doc.documentElement.getElementsByTagName("chapter")
for chapter in chapters:
    sections = chapter.getElementsByTagName("section")
    for section in sections:
        diags = section.getElementsByTagName("diag")
        for diag in diags:
            diagNameElement = diag.getElementsByTagName("name")[0]
            diagDescElement = diag.getElementsByTagName("desc")[0]
            df.append([diagNameElement.childNodes[0].data, diagDescElement.childNodes[0].data])
            subdiags = diag.getElementsByTagName("diag")
            for subdiag in subdiags:
                subdiagNameElement = subdiag.getElementsByTagName("name")[0]
                subdiagDescElement = subdiag.getElementsByTagName("desc")[0]
                df.append([subdiagNameElement.childNodes[0].data, subdiagDescElement.childNodes[0].data])
df = pd.DataFrame(df, columns = ['code', 'name']) # convert to pandas df
df = df.drop_duplicates() # drop any duplicate rows
df['load_ts'] = pd.Timestamp.utcnow() # add timestamp
df.head(10) # quick check result

,code,name,load_ts
0,A00,Cholera,2026-03-11 14:38:50.723985+00:00
1,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol...",2026-03-11 14:38:50.723985+00:00
2,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor",2026-03-11 14:38:50.723985+00:00
3,A00.9,"Cholera, unspecified",2026-03-11 14:38:50.723985+00:00
7,A01,Typhoid and paratyphoid fevers,2026-03-11 14:38:50.723985+00:00
8,A01.0,Typhoid fever,2026-03-11 14:38:50.723985+00:00
9,A01.00,"Typhoid fever, unspecified",2026-03-11 14:38:50.723985+00:00
10,A01.01,Typhoid meningitis,2026-03-11 14:38:50.723985+00:00
11,A01.02,Typhoid fever with heart involvement,2026-03-11 14:38:50.723985+00:00
12,A01.03,Typhoid pneumonia,2026-03-11 14:38:50.723985+00:00


In [4]:
# Save results to .csv file
df.to_csv('icd10_codes.csv', index = False) # save to .csv